In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

df_slacksize = df.copy()
# Map values in "slack" index level to 0.1 - "slack"
df_slacksize.reset_index("slack", inplace=True)
df_slacksize["slack"] = 0.1 - df_slacksize["slack"]
df_slacksize.set_index("slack", append=True, inplace=True)

palette = {"a": "#86f0af", "b": "#999999", "c": "#0a8339"}

for s in df.index.get_level_values("scenario").unique():
    sense = "opt"
    sns.lineplot(
        data=df_slacksize.xs(s, level="scenario").xs(sense, level="sense"),
        x="horizon",
        y="h2prod",
        hue="C",
        palette=palette,
        linewidth=3,
        ax=ax,
    )

# ax.set_ylim(0, 55)
ax.set_ylim(bottom=0)
ax.set_xlim(2025, 2050)

# Turn off x-grid
ax.xaxis.grid(False)

# Turn off spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

# Duplicate y ticks on right
ax.yaxis.set_tick_params(labelright=True)

# Set labels
ax.set_ylabel("H$_2$ production (Mt)")
ax.set_xlabel(None)

# Manually ad a legend with one entry for each pallete color
pretty_names = {"a": "Pessimistic CCS", "b": "Baseline", "c": "Optimistic CCS"}
handles = [
    plt.Line2D([0], [0], color=palette[k], label=pretty_names[k]) for k in palette
]
ax.legend(handles=handles, loc="upper left")

fig.savefig(
    "seminar/opt_h2_by_CCS.pdf", dpi=300, bbox_inches="tight", transparent=True
)

In [ ]:
# SEMINAR PLOT
fig, ax = plt.subplots(1, figsize=(18 * cm, 9 * cm))

quantiles = [1]
robust_colour = "#cdcb88"

df_copy = df.xs("b", level="B").copy()
plot_robust_ranges(df_copy, quantiles, 0.1, ax, plot_targets=False, max_y=38)

ax.set_ylabel("Robust green H$_2$ production [Mt]", fontsize=textsize)

fig.savefig(f"seminar/robustness.pdf", dpi=300, bbox_inches="tight", transparent=True)

plt.show()

In [ ]:
# H2 production

fig, ax = plt.subplots(figsize=(10, 6))

df_slacksize = df.copy()
# Map values in "slack" index level to 0.1 - "slack"
df_slacksize.reset_index("slack", inplace=True)
df_slacksize["slack"] = 0.1 - df_slacksize["slack"]
df_slacksize.set_index("slack", append=True, inplace=True)

palette = {"opt": "#114411", "min": "#4444ff55", "max": "#ff444455"}

for s in df.index.get_level_values("scenario").unique():
    sns.lineplot(data=df_slacksize.xs(s, level="scenario"), x="horizon", y="h2prod", hue="sense", palette=palette, size="slack", ax=ax)

# Increase zorder of all "opt" lines
for line in ax.lines:
    # Recognise by colour
    if line.get_color() == palette["opt"]:
        line.set_zorder(10)

ax.set_ylim(0, H2_MAX)
ax.set_xlim(2025, 2050)

# Turn off x-grid
ax.xaxis.grid(False)

# Turn off spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

# Duplicate y ticks on right
ax.yaxis.set_tick_params(labelright=True)

# Set labels
ax.set_ylabel("H$_2$ production (Mt)")
ax.set_xlabel(None)

# Add a single legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles[:9], labels=labels[:9], loc="upper left")

plt.show()

In [ ]:
# H2 storage over time
n = pypsa.Network("../results/Y1987_Bb/postnetworks/base_s_60_lc1.5__Ca-Ia-Eb_2050_max0.1.nc")

h2_stores_i = n.stores.index[n.stores.carrier == "H2 Store"]
h2_storage_e = n.stores_t.e[h2_stores_i].sum(axis=1) / 1e6

fig, ax = plt.subplots()
h2_storage_e.plot(ax=ax)
ax.set_ylim(bottom=0)

In [ ]:
# Plot storage levels by carreir
storage_e = n.stores_t.e.T.groupby(n.stores.carrier).sum().T / 1e6
# Select only columns of storage_e _not_ containing the "co2" substring or the "bio" substring
storage_e = storage_e.loc[:, ~storage_e.columns.str.contains("co2|bio|uranium")]

storage_unit_e = n.storage_units_t.state_of_charge.T.groupby(n.storage_units.carrier).sum().T / 1e6
storage_e = pd.concat([storage_e, storage_unit_e], axis=1)

# Filter out columns whose values are all below five
storage_e = storage_e.loc[:, (storage_e > 5).any()]

# Filter out columns whose values are constant (within 1)
storage_e = storage_e.loc[:, storage_e.std() > 1]

fig, ax = plt.subplots()
storage_e.plot(ax=ax, lw=2)

# Place legend outside of plot, three columns
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))

ax.set_ylim(bottom=0)